In [92]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
import phik
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss, brier_score_loss,
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, ConfusionMatrixDisplay,)
from sklearn.calibration import calibration_curve
import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
from optuna.trial import FixedTrial
import math
import category_encoders as ce


load_dotenv(find_dotenv(usecwd=True))
url = URL.create("postgresql+psycopg2",
    username=os.getenv("DB_USER"), password=os.getenv("DB_PASSWORD"),
    host="localhost", port=5433, database="hotel")
engine = create_engine(url)

df = pd.read_sql("SELECT * FROM hotels", engine)

print(df.shape)
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)
print(df['is_canceled'].mean())

(119390, 32)
(87396, 32)
0.27489816467572886


In [93]:
df["total_night"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df = df.drop(columns=["company","assigned_room_type","distribution_channel", "reservation_status", "reservation_status_date", "stays_in_weekend_nights", "stays_in_week_nights", "arrival_date_year", "arrival_date_month"])
df['country'] = df['country'].fillna("PRT")
df['agent'] = df['agent'].fillna(0)
df = df[df['lead_time'] < 600]
df = df[(df['adr'] < 5000) & (df['adr'] >= 0)]
df["adr_zero"] = (df["adr"] == 0).astype(int)
df = df[df['total_night'] < 31]
df["children"] = pd.to_numeric(df["children"], errors="coerce")
df = df.dropna(subset=["children"]).reset_index(drop=True)
df['children'] = df['children'].astype(int)


In [94]:
x = df.drop(columns='is_canceled')
y = df['is_canceled']

x_tmp, x_test, y_tmp, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
print(x_tmp.shape, y_tmp.shape)
x_tr, x_val, y_tr, y_val = train_test_split(x_tmp, y_tmp, test_size=0.25, random_state=42, stratify=y_tmp)


(69882, 24) (69882,)


60% train, 20% val, 20% test

In [95]:
order = ["No Deposit", "Non Refund", "Refundable"]
prep = ColumnTransformer(
    [
        ("hotel", OneHotEncoder(drop="if_binary", sparse_output=False), ["hotel"]),
        ("cust", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["customer_type"]),
        ("deposit", OrdinalEncoder(categories=[order], handle_unknown="use_encoded_value",
                                   unknown_value=-1), ["deposit_type"]),
        ("meal", ce.CountEncoder(normalize=True), ["meal"]),
        ("country", ce.BinaryEncoder(), ["country"]),
        ("market_segment", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["market_segment"]),
        ("reserved_room_type", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["reserved_room_type"]),

    ],
    remainder="passthrough",
)

In [96]:
prep.set_params(verbose_feature_names_out=False)
prep.set_output(transform="pandas")

x_tr_t = prep.fit_transform(x_tr)
x_val_t = prep.transform(x_val)
x_test_t = prep.transform(x_test)
print(x_tr_t.head(5).T)
print(x_val_t.shape)

                                    59012     26438       30694      31027  \
hotel_Resort Hotel               0.000000   1.00000    1.000000    1.00000   
customer_type_Contract           0.000000   0.00000    0.000000    0.00000   
customer_type_Group              0.000000   0.00000    0.000000    0.00000   
customer_type_Transient          1.000000   1.00000    1.000000    1.00000   
customer_type_Transient-Party    0.000000   0.00000    0.000000    0.00000   
deposit_type                     0.000000   0.00000    0.000000    0.00000   
meal                             0.107535   0.77644    0.106428    0.77644   
country_0                        0.000000   0.00000    0.000000    0.00000   
country_1                        0.000000   0.00000    0.000000    0.00000   
country_2                        0.000000   0.00000    0.000000    0.00000   
country_3                        0.000000   0.00000    0.000000    0.00000   
country_4                        0.000000   0.00000    0.000000 

In [97]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(x_tr, y_tr)
print(roc_auc_score(y_test, dummy.predict_proba(x_test)[:, 1]))
print(accuracy_score(y_test, dummy.predict(x_test)))

0.5
0.7252017629214126


baseline - most_frequent

In [99]:
print(df.groupby("children")["is_canceled"].agg(n="count", cancel_share="mean").round(3))

              n  cancel_share
children                     
0         78989         0.266
1          4695         0.312
2          3593         0.418
3            75         0.213
10            1         1.000


очень старнная аномалия, в 95% людей с невозратным дипозитом отменяют бронь, пока я оставлю этот бред, потом удалю при первой возможности